# Your pipeline is green. Finance says revenue is wrong. Find the bug in 90 minutes. You won't, the first time.

A finance analyst pings you at 9 AM. "Revenue is off by a few hundred bucks. Pipeline says green." You open CloudWatch and every alarm is happy. You open the pipeline dashboard and every box is checked. Something is wrong but nothing is screaming.

Today you build the diagnostic muscle for the case where the system is lying to you. The bug is real: the consumer Lambda derives a synthetic `event_id` from `hash(event_timestamp, customer_id)` instead of using the producer-emitted `event_id`. Under a high write rate with Lambda concurrency capped at 1, the hash collides for roughly 0.3 percent of records, the DynamoDB conditional write skips them as "duplicates", and the sink quietly drops about 30 records per 10,000.

The four deliverables map to four checkpoints:
1. Pre-broken environment is up; producer log shows 10,000 records emitted.
2. Diagnostic phase: you ran at least three CloudWatch Logs Insights queries against the consumer log group before fixing anything.
3. Bug identified and fixed: redeploy the consumer; new producer run; sink contains 10,000 with no drops.
4. Trim-horizon replay over a 5-minute window: zero gap between stream-side event_ids and sink-side event_ids for the recent batch.

**Time.** About 90 minutes hands-on. The first 60 minutes are diagnostic; the last 30 are the fix + verification.

**Cost.** This lab costs about $1.20 on a clean run. Kinesis 1 shard at $0.015/hr is the steady drain. Two producer runs against the Iceberg sink (one to surface the bug, one to confirm the fix) is the main spend; about $0.40 in Iceberg writes. Athena scans for diagnostic queries and parity are pennies. Realistic upper bound: $2.50.

**Critical resource.** The Kinesis stream is critical-tagged. Cleanup tears it down first.

This lab is part of the Labrun Data Engineering job-prep track, Pipeline Engineering sub-track. It is a killer lab and a chaos-mode lab.


In [ ]:
!pip install --quiet labrun-checks==0.7.0 boto3


In [ ]:
# Imports and constants.

import atexit
import base64
import io
import json
import re
import sys
import time
import uuid
import zipfile
from getpass import getpass

import boto3
from botocore.exceptions import ClientError

from labrun_checks import CheckpointResult, CleanupResource, check, cleanup, register, run_cleanup

LAB_SLUG = "silent-dropper-pipeline-debug"
LAB_ID = LAB_SLUG
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_SLUG
REGION = "us-east-1"

BUCKET_NAME = None
STREAM_NAME = f"labrun-{LAB_SLUG}-stream"
WATERMARK_TABLE = f"labrun-{LAB_SLUG}-watermark"
CONSUMER_ROLE = f"labrun-{LAB_SLUG}-consumer-role"
CONSUMER_FN = f"labrun-{LAB_SLUG}-consumer"
DATABASE_NAME = f"labrun_{LAB_SLUG.replace('-', '_')}_db"
SINK_TABLE = "revenue_iceberg"
ATHENA_WG = f"labrun-{LAB_SLUG}-wg"

ACCOUNT_ID = None
SESSION_START_EPOCH = int(time.time())
PRODUCER_RUN_BATCHES = []


In [ ]:
# NBVAL_SKIP
# Setup.

print("Paste your lab session token:")
SESSION_TOKEN = getpass("Session token: ").strip()
if not SESSION_TOKEN:
    raise SystemExit("Missing session token.")

AWS_ACCESS_KEY_ID = getpass("AWS_ACCESS_KEY_ID: ").strip()
AWS_SECRET_ACCESS_KEY = getpass("AWS_SECRET_ACCESS_KEY: ").strip()
AWS_SESSION_TOKEN = getpass("AWS_SESSION_TOKEN (Enter to skip): ").strip() or None

AWS_CREDENTIALS = {"aws_access_key_id": AWS_ACCESS_KEY_ID, "aws_secret_access_key": AWS_SECRET_ACCESS_KEY, "aws_session_token": AWS_SESSION_TOKEN}
sts = boto3.client("sts", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
try:
    ACCOUNT_ID = sts.get_caller_identity()["Account"]
except ClientError as exc:
    raise SystemExit(f"AWS credentials rejected: {exc}")

BUCKET_NAME = f"labrun-{LAB_SLUG}-{ACCOUNT_ID}"
print(f"Account: {ACCOUNT_ID}; bucket: {BUCKET_NAME}")
register(SESSION_TOKEN, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print("Setup complete.")


In [ ]:
# NBVAL_SKIP
# CLEANUP_MANIFEST + atexit + orphan scan.

CLEANUP_MANIFEST = [
    CleanupResource(type="kinesis_stream", id=STREAM_NAME, region=REGION, critical=True,
        cli_delete_command=f"aws kinesis delete-stream --stream-name {STREAM_NAME} --enforce-consumer-deletion --region {REGION}"),
    CleanupResource(type="lambda_event_source_mapping", id="pending", region=REGION),
    CleanupResource(type="lambda_function", id=CONSUMER_FN, region=REGION,
        cli_delete_command=f"aws lambda delete-function --function-name {CONSUMER_FN} --region {REGION}"),
    CleanupResource(type="dynamodb_table", id=WATERMARK_TABLE, region=REGION,
        cli_delete_command=f"aws dynamodb delete-table --table-name {WATERMARK_TABLE} --region {REGION}"),
    CleanupResource(type="iceberg_table", id=f"{DATABASE_NAME}.{SINK_TABLE}", region=REGION),
    CleanupResource(type="iam_role", id=CONSUMER_ROLE, region=REGION),
    CleanupResource(type="athena_workgroup", id=ATHENA_WG, region=REGION,
        cli_delete_command=f"aws athena delete-work-group --work-group {ATHENA_WG} --recursive-delete-option --region {REGION}"),
    CleanupResource(type="glue_database", id=DATABASE_NAME, region=REGION),
    CleanupResource(type="s3_bucket", id=BUCKET_NAME, region=REGION),
]


def _atexit_cleanup():
    print("[atexit] Kinesis bills hourly while alive. Run the cleanup cell if it has not run.")


atexit.register(_atexit_cleanup)

tagging = boto3.client("resourcegroupstaggingapi", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
try:
    orphan_arns = [r["ResourceARN"] for r in tagging.get_resources(TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}]).get("ResourceTagMappingList", [])]
except ClientError as exc:
    raise SystemExit(f"Orphan scan failed: {exc}")

if orphan_arns:
    print("Found orphans. Cleanup is blocking.")
    for arn in orphan_arns:
        print(f"  ORPHAN: {arn}")
    raise SystemExit("Run previous session's cleanup cell first.")

print("No orphans. Manifest declared.")


## Standing up the broken environment

The next cell stands up the pipeline with the deterministic bug. The consumer Lambda is reserved-concurrency-capped at 1, which keeps the hash-collision rate stable across runs. The producer emits 10,000 records with deterministic event_timestamps; certain millisecond buckets repeat by design.

You do not edit this cell. After it runs you have:
- 1-shard Kinesis stream (the producer surface)
- DynamoDB on-demand watermark table (the dedup primitive the consumer abuses)
- Iceberg sink `revenue_iceberg`
- Lambda consumer with the bug (reserved concurrency = 1)
- Event source mapping wired with BatchSize=100, StartingPosition=TRIM_HORIZON


In [ ]:
# NBVAL_SKIP
# Setup helper: stand up the broken pipeline.

s3 = boto3.client("s3", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
iam = boto3.client("iam", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
glue = boto3.client("glue", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
ddb = boto3.client("dynamodb", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
kinesis = boto3.client("kinesis", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
lambda_client = boto3.client("lambda", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
athena = boto3.client("athena", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
logs = boto3.client("logs", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})


def stand_up_broken_environment():
    global ESM_UUID
    try:
        s3.create_bucket(Bucket=BUCKET_NAME)
        s3.put_bucket_tagging(Bucket=BUCKET_NAME, Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]})
    except ClientError as exc:
        if exc.response["Error"]["Code"] != "BucketAlreadyOwnedByYou":
            raise
    try:
        glue.create_database(DatabaseInput={"Name": DATABASE_NAME})
    except glue.exceptions.AlreadyExistsException:
        pass
    try:
        athena.create_work_group(
            Name=ATHENA_WG,
            Configuration={"ResultConfiguration": {"OutputLocation": f"s3://{BUCKET_NAME}/athena/"}, "EnforceWorkGroupConfiguration": True},
            Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
        )
    except Exception:
        pass

    def run_athena(query, timeout=60):
        qid = athena.start_query_execution(QueryString=query, WorkGroup=ATHENA_WG, QueryExecutionContext={"Database": DATABASE_NAME})["QueryExecutionId"]
        deadline = time.time() + timeout
        while time.time() < deadline:
            s = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]["State"]
            if s == "SUCCEEDED":
                return qid
            if s in ("FAILED", "CANCELLED"):
                raise RuntimeError(f"Athena {qid} {s}")
            time.sleep(2)
        raise TimeoutError(f"Athena {qid} timed out")

    run_athena(
        f"CREATE TABLE IF NOT EXISTS {DATABASE_NAME}.{SINK_TABLE} ("
        f"  event_id string, customer_id string, amount_cents bigint, "
        f"  event_timestamp string, producer_batch_id string, derived_id string"
        f") LOCATION 's3://{BUCKET_NAME}/{SINK_TABLE}/' "
        f"TBLPROPERTIES ('table_type'='ICEBERG', 'format'='parquet')"
    )

    try:
        kinesis.create_stream(StreamName=STREAM_NAME, ShardCount=1, StreamModeDetails={"StreamMode": "PROVISIONED"}, Tags={LAB_TAG_KEY: LAB_TAG_VALUE})
    except kinesis.exceptions.ResourceInUseException:
        pass
    print("  stream creating, hold ~60s...")
    deadline = time.time() + 180
    while time.time() < deadline:
        try:
            if kinesis.describe_stream_summary(StreamName=STREAM_NAME)["StreamDescriptionSummary"]["StreamStatus"] == "ACTIVE":
                break
        except Exception:
            pass
        time.sleep(5)
    stream_arn = f"arn:aws:kinesis:{REGION}:{ACCOUNT_ID}:stream/{STREAM_NAME}"

    try:
        ddb.create_table(
            TableName=WATERMARK_TABLE,
            AttributeDefinitions=[{"AttributeName": "event_id", "AttributeType": "S"}],
            KeySchema=[{"AttributeName": "event_id", "KeyType": "HASH"}],
            BillingMode="PAY_PER_REQUEST",
            Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
        )
        ddb.get_waiter("table_exists").wait(TableName=WATERMARK_TABLE)
    except ddb.exceptions.ResourceInUseException:
        pass

    try:
        iam.create_role(
            RoleName=CONSUMER_ROLE,
            AssumeRolePolicyDocument=json.dumps({"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"}, "Action": "sts:AssumeRole"}]}),
            Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
        )
        iam.put_role_policy(RoleName=CONSUMER_ROLE, PolicyName="labrun-consumer-inline", PolicyDocument=json.dumps({
            "Version": "2012-10-17",
            "Statement": [{"Effect": "Allow", "Action": ["kinesis:*", "dynamodb:*", "logs:*", "athena:*", "glue:*", "s3:*"], "Resource": "*"}],
        }))
    except iam.exceptions.EntityAlreadyExistsException:
        pass
    time.sleep(15)
    role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{CONSUMER_ROLE}"

    BROKEN_CONSUMER_CODE = f'''
import base64
import hashlib
import json
import time
import boto3

DDB = boto3.client("dynamodb", region_name="{REGION}")
ATHENA = boto3.client("athena", region_name="{REGION}")
WATERMARK = "{WATERMARK_TABLE}"
DB = "{DATABASE_NAME}"
TABLE = "{SINK_TABLE}"
WG = "{ATHENA_WG}"


def derive_event_id(record):
    """BUG: rebuilds event_id from a hash; collisions drop records."""
    key = (record.get("event_timestamp", "") + "|" + record.get("customer_id", "")).encode()
    return hashlib.sha256(key).hexdigest()[:16]


def _run_athena(query, timeout_s=30):
    qid = ATHENA.start_query_execution(QueryString=query, WorkGroup=WG, QueryExecutionContext={{"Database": DB}})["QueryExecutionId"]
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        s = ATHENA.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]["State"]
        if s == "SUCCEEDED":
            return
        if s in ("FAILED", "CANCELLED"):
            raise RuntimeError(f"Athena {{qid}} {{s}}")
        time.sleep(1)
    raise TimeoutError(f"Athena {{qid}} timed out")


def _q(v):
    if v is None:
        return "NULL"
    if isinstance(v, (int, float)):
        return str(v)
    return "'" + str(v).replace("'", "''") + "'"


def handler(event, context):
    inserts = []
    dropped = 0
    for r in event.get("Records", []):
        data = json.loads(base64.b64decode(r["kinesis"]["data"]))
        derived = derive_event_id(data)
        try:
            DDB.put_item(
                TableName=WATERMARK,
                Item={{"event_id": {{"S": derived}}}},
                ConditionExpression="attribute_not_exists(event_id)",
            )
        except DDB.exceptions.ConditionalCheckFailedException:
            dropped += 1
            continue
        inserts.append(
            f"({{_q(data.get('event_id'))}}, {{_q(data.get('customer_id'))}}, "
            f"{{int(data.get('amount_cents', 0))}}, {{_q(data.get('event_timestamp'))}}, "
            f"{{_q(data.get('producer_batch_id'))}}, {{_q(derived)}})"
        )
    if inserts:
        _run_athena(
            f"INSERT INTO {{DB}}.{{TABLE}} "
            f"(event_id, customer_id, amount_cents, event_timestamp, producer_batch_id, derived_id) "
            f"VALUES " + ", ".join(inserts),
            timeout_s=60,
        )
    print(f"processed={{len(inserts)}} dropped_as_duplicate={{dropped}}")
    return {{"processed": len(inserts), "dropped": dropped}}
'''

    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("index.py", BROKEN_CONSUMER_CODE)
    buf.seek(0)
    zip_bytes = buf.read()

    for attempt in range(8):
        try:
            lambda_client.create_function(
                FunctionName=CONSUMER_FN, Runtime="python3.13", Role=role_arn,
                Handler="index.handler", Code={"ZipFile": zip_bytes},
                Timeout=180, MemorySize=512, Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
            )
            break
        except lambda_client.exceptions.InvalidParameterValueException:
            time.sleep(5)
        except lambda_client.exceptions.ResourceConflictException:
            break

    # Cap concurrency to surface the bug.
    try:
        lambda_client.put_function_concurrency(FunctionName=CONSUMER_FN, ReservedConcurrentExecutions=1)
    except Exception:
        pass

    esm = lambda_client.create_event_source_mapping(
        EventSourceArn=stream_arn, FunctionName=CONSUMER_FN,
        StartingPosition="TRIM_HORIZON", BatchSize=100,
        MaximumBatchingWindowInSeconds=2, Enabled=True,
    )
    ESM_UUID = esm["UUID"]
    for r in CLEANUP_MANIFEST:
        if r.type == "lambda_event_source_mapping":
            r.id = ESM_UUID
            break

    print("broken pipeline up; reserved concurrency = 1 for the consumer (this is the trap).")


stand_up_broken_environment()


def run_producer(batch_id, n=10000):
    """Emit n records with deterministic event_timestamps that create collisions."""
    import random
    rnd = random.Random(hash(batch_id) % 2**32)
    base_epoch = int(time.time()) - 600
    # Restrict event_timestamp resolution to 30s buckets to force collisions.
    bucket_count = 250
    records = [
        {
            "event_id": str(uuid.uuid4()),
            "customer_id": f"cust-{rnd.randint(1, 200)}",
            "amount_cents": rnd.randint(100, 10000),
            "event_timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime(base_epoch + (i % bucket_count) * 30)),
            "producer_batch_id": batch_id,
        }
        for i in range(n)
    ]
    # Put in chunks of 500.
    for off in range(0, n, 500):
        kinesis.put_records(StreamName=STREAM_NAME, Records=[
            {"Data": json.dumps(r).encode("utf-8"), "PartitionKey": r["customer_id"]} for r in records[off:off + 500]
        ])
    s3.put_object(Bucket=BUCKET_NAME, Key=f"producer/{batch_id}.json", Body=json.dumps({"batch_id": batch_id, "emitted": len(records)}).encode("utf-8"))
    return records


PRODUCER_BATCH_1 = f"batch-{int(time.time())}-1"
PRODUCER_RUN_BATCHES.append(PRODUCER_BATCH_1)
run_producer(PRODUCER_BATCH_1)
print(f"Producer batch 1 emitted (10,000 records): {PRODUCER_BATCH_1}")
print("Drain wait ~5 minutes for Lambda (concurrency=1) to process...")
time.sleep(300)


In [ ]:
# NBVAL_SKIP
# Checkpoint 1: producer log shows 10,000 records.

def validator_1(session):
    try:
        log = json.loads(s3.get_object(Bucket=BUCKET_NAME, Key=f"producer/{PRODUCER_BATCH_1}.json")["Body"].read())
    except ClientError as exc:
        return CheckpointResult("fail", error_reason=f"Producer log missing: {exc}")
    if log.get("emitted") != 10000:
        return CheckpointResult("fail", error_reason=f"emitted={log.get('emitted')}, expected 10000")
    try:
        if kinesis.describe_stream_summary(StreamName=STREAM_NAME)["StreamDescriptionSummary"]["StreamStatus"] != "ACTIVE":
            return CheckpointResult("fail", error_reason=f"Stream not ACTIVE")
    except kinesis.exceptions.ResourceNotFoundException:
        return CheckpointResult("fail", error_reason=f"Stream {STREAM_NAME} missing")
    return CheckpointResult("pass")


check(1, validator_1)


<details><summary>Hint 1 (nudge)</summary>

The producer ran during setup. The validator just confirms the log file is on S3 and the stream is alive.

</details>

<details><summary>Hint 2 (direction)</summary>

If the validator fails on "Stream not ACTIVE", wait another 30 seconds and retry. The stream needs a minute or two after creation to become available.

</details>

<details><summary>Hint 3 (spoiler)</summary>

No code to write. Just run the checkpoint cell.

</details>


**Wallet check.** Stream has been alive ~10 minutes: under 1 cent. Producer S3 log: free. First Athena INSERTs from the consumer: a few cents. Session total so far: ~5 cents.

## Task 2: Diagnostic phase. Run at least three CloudWatch Logs Insights queries before fixing anything.

The pipeline looks green. Stream metrics show records in and records out matched. Sink count looks reasonable. CloudWatch alarms are quiet. Where do you look?

The lab requires you to do real diagnostic work before the fix path opens. The checkpoint validates that you ran at least three Logs Insights queries against the consumer log group. It does not score which queries; only that diagnostic work happened.

Suggested starting queries (you can run all three or invent your own):

1. `fields @timestamp, @message | filter @message like /ERROR/`
2. `fields @timestamp, @message | filter @message like /dropped_as_duplicate/ | stats sum(dropped_as_duplicate) as total_dropped`
3. `fields @timestamp, @message | stats count() as inv_count by bin(1m)`


In [ ]:
# NBVAL_SKIP
# Task 2: Run at least 3 CloudWatch Logs Insights queries.

# YOUR CODE: Use logs.start_query() and poll until COMPLETE. Run at least
# three queries with different filter or stats expressions. Each query
# must have startTime >= SESSION_START_EPOCH so the validator can see it.

# Example helper:
def run_cwli(query_string, log_group=f"/aws/lambda/{CONSUMER_FN}", minutes=30):
    qid = logs.start_query(
        logGroupName=log_group,
        startTime=int(time.time()) - minutes * 60,
        endTime=int(time.time()),
        queryString=query_string,
    )["queryId"]
    deadline = time.time() + 60
    while time.time() < deadline:
        resp = logs.get_query_results(queryId=qid)
        if resp["status"] == "Complete":
            return resp["results"]
        if resp["status"] in ("Failed", "Cancelled", "Timeout"):
            return []
        time.sleep(2)
    return []


# YOUR CODE: invoke run_cwli at least three times with different queries.


In [ ]:
# NBVAL_SKIP
# Checkpoint 2: describe_queries returns at least 3 distinct queries for the consumer log group.

def validator_2(session):
    log_group = f"/aws/lambda/{CONSUMER_FN}"
    try:
        resp = logs.describe_queries(logGroupName=log_group, maxResults=50)
    except logs.exceptions.ResourceNotFoundException:
        return CheckpointResult("fail", error_reason=f"Log group {log_group} not found yet. The consumer must have invoked at least once to create it.")
    queries = resp.get("queries", [])
    in_session = [q for q in queries if q.get("createTime", 0) // 1000 >= SESSION_START_EPOCH - 5]
    distinct = set((q.get("queryString") or "").strip() for q in in_session)
    if len(distinct) < 3:
        return CheckpointResult("fail", error_reason=f"Found {len(distinct)} distinct CWLI queries since SESSION_START_EPOCH; need at least 3. Run more diagnostic queries before claiming the bug is identified.")
    return CheckpointResult("pass")


check(2, validator_2)


<details><summary>Hint 1 (nudge)</summary>

Start with `filter @message like /ERROR/`. You probably will not find anything; that is the lab. The bug is not surfacing as an error.

</details>

<details><summary>Hint 2 (direction)</summary>

The consumer logs `processed=N dropped_as_duplicate=M` on every invocation. Sum the dropped counter across the run window. Compare against the producer's emitted count. The gap is your signal.

Query that surfaces the bug:

```
fields @timestamp, @message
| parse @message "dropped_as_duplicate=*" as dropped
| stats sum(dropped) as total_dropped
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
run_cwli('fields @timestamp, @message | filter @message like /ERROR/')
run_cwli('fields @timestamp, @message | parse @message "dropped_as_duplicate=*" as dropped | stats sum(dropped) as total_dropped')
run_cwli('fields @timestamp, @message | parse @message "processed=*" as processed | stats sum(processed) as total_processed')
```

</details>


**Wallet check.** CloudWatch Logs Insights is $0.005 per GB scanned. Three queries against a small log group: a fraction of a cent. Session total so far: ~5 cents.

## Task 3: Identify the bug, redeploy the consumer, prove zero drops on a fresh batch

The bug is in `derive_event_id`. The consumer rebuilds an event_id from `(event_timestamp, customer_id)` hash, ignoring the producer's actual `event_id` field. At high write rate with the producer's coarse event_timestamp bucketing, two distinct records collide on the hash; the DynamoDB conditional write rejects the second one as a duplicate.

The fix: use the producer's `event_id` directly. Three lines.

You read the current Lambda code, replace the bug, redeploy, run a fresh producer batch, wait for the drain, assert zero drops.


In [ ]:
# NBVAL_SKIP
# Task 3: Fix the consumer, redeploy, run a fresh batch, prove no drops.

current_code = lambda_client.get_function(FunctionName=CONSUMER_FN)["Code"]["Location"]
# Easier: rebuild the function source from scratch with the fix.

FIXED_CONSUMER_CODE = f'''
import base64
import json
import time
import boto3

DDB = boto3.client("dynamodb", region_name="{REGION}")
ATHENA = boto3.client("athena", region_name="{REGION}")
WATERMARK = "{WATERMARK_TABLE}"
DB = "{DATABASE_NAME}"
TABLE = "{SINK_TABLE}"
WG = "{ATHENA_WG}"


def derive_event_id(record):
    # FIX: use the producer-emitted event_id directly. The producer guarantees
    # uniqueness; rebuilding from a hash creates collisions.
    return record["event_id"]


def _run_athena(query, timeout_s=30):
    qid = ATHENA.start_query_execution(QueryString=query, WorkGroup=WG, QueryExecutionContext={{"Database": DB}})["QueryExecutionId"]
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        s = ATHENA.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]["State"]
        if s == "SUCCEEDED":
            return
        if s in ("FAILED", "CANCELLED"):
            raise RuntimeError(f"Athena {{qid}} {{s}}")
        time.sleep(1)
    raise TimeoutError(f"Athena {{qid}} timed out")


def _q(v):
    if v is None:
        return "NULL"
    if isinstance(v, (int, float)):
        return str(v)
    return "'" + str(v).replace("'", "''") + "'"


def handler(event, context):
    inserts = []
    dropped = 0
    for r in event.get("Records", []):
        data = json.loads(base64.b64decode(r["kinesis"]["data"]))
        derived = derive_event_id(data)
        try:
            DDB.put_item(
                TableName=WATERMARK, Item={{"event_id": {{"S": derived}}}},
                ConditionExpression="attribute_not_exists(event_id)",
            )
        except DDB.exceptions.ConditionalCheckFailedException:
            dropped += 1
            continue
        inserts.append(
            f"({{_q(data.get('event_id'))}}, {{_q(data.get('customer_id'))}}, "
            f"{{int(data.get('amount_cents', 0))}}, {{_q(data.get('event_timestamp'))}}, "
            f"{{_q(data.get('producer_batch_id'))}}, {{_q(derived)}})"
        )
    if inserts:
        _run_athena(
            f"INSERT INTO {{DB}}.{{TABLE}} (event_id, customer_id, amount_cents, event_timestamp, producer_batch_id, derived_id) VALUES "
            + ", ".join(inserts),
            timeout_s=60,
        )
    print(f"processed={{len(inserts)}} dropped_as_duplicate={{dropped}}")
    return {{"processed": len(inserts), "dropped": dropped}}
'''


def _zip(code):
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("index.py", code)
    buf.seek(0)
    return buf.read()


# YOUR CODE: Deploy the fixed code with
#   lambda_client.update_function_code(FunctionName=CONSUMER_FN, ZipFile=_zip(FIXED_CONSUMER_CODE))
# Wait ~10s for the update to settle.

# YOUR CODE: Run a fresh producer batch (different batch_id) so the new
# code path executes against new traffic. The records already on the
# stream from setup were processed by the BROKEN code.
#   PRODUCER_BATCH_2 = f"batch-{int(time.time())}-2"
#   PRODUCER_RUN_BATCHES.append(PRODUCER_BATCH_2)
#   run_producer(PRODUCER_BATCH_2)
#   time.sleep(300)  # ~5 min drain


In [ ]:
# NBVAL_SKIP
# Checkpoint 3: sink contains exactly 10,000 rows for the fresh batch_id.

def validator_3(session):
    if len(PRODUCER_RUN_BATCHES) < 2:
        return CheckpointResult("fail", error_reason="Need a fresh producer batch after the fix. Append a new batch_id to PRODUCER_RUN_BATCHES and run the producer.")
    latest = PRODUCER_RUN_BATCHES[-1]
    qid = athena.start_query_execution(
        QueryString=f"SELECT count(*) FROM {DATABASE_NAME}.{SINK_TABLE} WHERE producer_batch_id = '{latest}'",
        WorkGroup=ATHENA_WG, QueryExecutionContext={"Database": DATABASE_NAME},
    )["QueryExecutionId"]
    deadline = time.time() + 60
    while time.time() < deadline:
        ex = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]
        if ex["Status"]["State"] == "SUCCEEDED":
            rows = athena.get_query_results(QueryExecutionId=qid)["ResultSet"]["Rows"]
            count = int(rows[1]["Data"][0]["VarCharValue"])
            if count != 10000:
                return CheckpointResult("fail", error_reason=f"Sink contains {count} rows for the fresh batch, expected 10000. Wait another 60s for drain and re-run.")
            return CheckpointResult("pass")
        if ex["Status"]["State"] in ("FAILED", "CANCELLED"):
            return CheckpointResult("error", error_reason=f"Athena query {ex['Status']['State']}")
        time.sleep(2)
    return CheckpointResult("error", error_reason="Athena query timed out")


check(3, validator_3)


<details><summary>Hint 1 (nudge)</summary>

The bug is in the consumer's dedup path, not the stream or the sink. The drops happen when the conditional write rejects a "duplicate" that is not actually a duplicate.

</details>

<details><summary>Hint 2 (direction)</summary>

`derive_event_id` rebuilds the id from `(event_timestamp, customer_id)`. Two distinct records can land in the same 30-second timestamp bucket with the same customer_id. The producer already emits a unique `event_id` field; use it directly.

</details>

<details><summary>Hint 3 (spoiler)</summary>

The fix is a 1-line change inside the Lambda:

```python
def derive_event_id(record):
    return record["event_id"]
```

Then redeploy and run a new producer batch:

```python
lambda_client.update_function_code(FunctionName=CONSUMER_FN, ZipFile=_zip(FIXED_CONSUMER_CODE))
time.sleep(10)
PRODUCER_BATCH_2 = f"batch-{int(time.time())}-2"
PRODUCER_RUN_BATCHES.append(PRODUCER_BATCH_2)
run_producer(PRODUCER_BATCH_2)
time.sleep(300)
```

</details>


**Wallet check.** Second producer + drain: ~$0.40 in Iceberg writes plus a fraction of a cent for Athena. Session total so far: ~50 cents.

## Task 4: Trim-horizon replay over the recent batch. Zero gap.

The fix is in. The sink count matches the producer count. Last check: read directly from the stream's trim-horizon over the recent 5-minute window, collect every event_id, and diff against the sink for the same producer_batch_id. Zero gap is the goal.


In [ ]:
# NBVAL_SKIP
# Task 4: Trim-horizon replay confirms zero gap.

def trim_horizon_replay(batch_id, max_records=15000):
    """Read every record on the stream for this batch_id; return set of event_ids."""
    shards = kinesis.list_shards(StreamName=STREAM_NAME)["Shards"]
    seen = set()
    for shard in shards:
        it_resp = kinesis.get_shard_iterator(
            StreamName=STREAM_NAME, ShardId=shard["ShardId"], ShardIteratorType="TRIM_HORIZON",
        )
        iterator = it_resp["ShardIterator"]
        for _ in range(200):
            r = kinesis.get_records(ShardIterator=iterator, Limit=1000)
            for rec in r["Records"]:
                data = json.loads(rec["Data"])
                if data.get("producer_batch_id") == batch_id:
                    seen.add(data["event_id"])
            iterator = r.get("NextShardIterator")
            if not iterator or len(r["Records"]) == 0:
                break
            if len(seen) >= max_records:
                break
    return seen


# YOUR CODE: call trim_horizon_replay on the latest batch_id, query the
# sink for the same batch_id, compute the set difference. Store the gap
# size in REPLAY_GAP (an int; 0 is the expected pass state).


In [ ]:
# NBVAL_SKIP
# Checkpoint 4: REPLAY_GAP == 0.

def validator_4(session):
    gap = globals().get("REPLAY_GAP")
    if gap is None:
        return CheckpointResult("fail", error_reason="REPLAY_GAP not set. Run trim_horizon_replay and the sink query, then assign the set difference size to REPLAY_GAP.")
    if gap != 0:
        return CheckpointResult("fail", error_reason=f"REPLAY_GAP is {gap}; expected 0. The stream still has event_ids the sink does not.")
    return CheckpointResult("pass")


check(4, validator_4)


<details><summary>Hint 1 (nudge)</summary>

Read from trim-horizon for the latest batch only. Then diff the stream-side set against the sink-side set.

</details>

<details><summary>Hint 2 (direction)</summary>

`trim_horizon_replay(batch_id)` returns a set of event_ids from the stream. Query the sink for the same batch_id; convert to a set; the difference (`stream_set - sink_set`) is your gap.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
latest = PRODUCER_RUN_BATCHES[-1]
stream_set = trim_horizon_replay(latest)
qid = athena.start_query_execution(
    QueryString=f"SELECT event_id FROM {DATABASE_NAME}.{SINK_TABLE} WHERE producer_batch_id = '{latest}'",
    WorkGroup=ATHENA_WG, QueryExecutionContext={"Database": DATABASE_NAME},
)["QueryExecutionId"]
deadline = time.time() + 120
while time.time() < deadline:
    ex = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]
    if ex["Status"]["State"] == "SUCCEEDED":
        break
    if ex["Status"]["State"] in ("FAILED", "CANCELLED"):
        raise RuntimeError("Athena failed")
    time.sleep(2)
sink_set = set()
paginator = athena.get_paginator("get_query_results")
for page in paginator.paginate(QueryExecutionId=qid):
    for r in page["ResultSet"]["Rows"][1:]:
        sink_set.add(r["Data"][0]["VarCharValue"])
REPLAY_GAP = len(stream_set - sink_set)
print(f"Stream: {len(stream_set)}; sink: {len(sink_set)}; gap: {REPLAY_GAP}")
```

</details>


**Wallet check.** Two Athena scans for the parity diff: under 1 cent. Kinesis GetRecords: free. Session total so far: ~50 cents.

In [ ]:
# NBVAL_SKIP
from labrun_checks.adapters.aws import AwsCleanupAdapter


class _LabAdapter(AwsCleanupAdapter):
    def delete_resource(self, credentials, resource):
        if resource.type == "iceberg_table":
            db, table = resource.id.split(".", 1)
            client = boto3.client("glue", region_name=resource.region, **{k: v for k, v in credentials.items() if v})
            try:
                client.delete_table(DatabaseName=db, Name=table)
            except client.exceptions.EntityNotFoundException:
                pass
            return
        if resource.type == "athena_workgroup":
            client = boto3.client("athena", region_name=resource.region, **{k: v for k, v in credentials.items() if v})
            try:
                client.delete_work_group(WorkGroup=resource.id, RecursiveDeleteOption=True)
            except Exception:
                pass
            return
        return super().delete_resource(credentials, resource)

    def describe_resource(self, credentials, resource):
        if resource.type in ("iceberg_table", "athena_workgroup"):
            return False
        return super().describe_resource(credentials, resource)


result = run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter())

print()
print("Cleanup complete.")
critical_count = sum(1 for r in CLEANUP_MANIFEST if r.critical)
standard_count = len(CLEANUP_MANIFEST) - critical_count
print(f"Critical resources destroyed: {critical_count}")
print(f"Standard resources destroyed: {standard_count}")
failed = len(result.warnings) + len(result.orphans)
print(f"Resources that failed to delete: {failed}")
if failed > 0:
    print("If failed > 0, your AWS account may still incur charges. Resolve before closing this notebook.")
    for w in result.warnings:
        print(f"  WARNING: {w}")
    for o in result.orphans:
        print(f"  ORPHAN: {o}")

cleanup(status=result.status)
if result.status == "dirty":
    sys.exit(1)


**Session total: $1.20 to $2.50.** Two producer runs against Iceberg were the main spend. Kinesis shard-hour ticked across the 90+ min session. CloudWatch Logs Insights queries: pennies. The diagnostic phase is where the lesson lives; the fix is trivial once the diagnosis is right.

## These are not graded. They are for you.

**1. Diagnostic path.** Walk through your queries. What was the first signal that something was wrong? What would you add to the pipeline's observability so a silent 0.3% drop would alarm the next time?

**2. The structural alternative.** The bug existed because the consumer reused the dedup primitive (DynamoDB conditional write) on a derived key. What is the lighter-weight alternative when the producer already emits unique event_ids? Walk through the trade-offs of dropping DynamoDB entirely versus keeping it as a defense-in-depth.

## Exam-Style Practice

**Question 1.** Your pipeline metrics look healthy (record counts match, no error logs, no failed invocations) but the downstream revenue sum is wrong by a small percentage. The most likely class of bug is:

A) Schema drift between producer and consumer.

B) Silent record drops in a consumer with weak deduplication logic.

C) Clock skew between producer and consumer.

D) Producer batching that delays records past the sink's read window.

<details><summary>Show answer</summary>

**B**.

- A) Wrong. Schema drift typically surfaces as parse errors or field-not-found exceptions in the consumer logs. The lab's hypothetical says "no error logs", which rules schema drift out.
- B) Right. The lab's bug is exactly this: the consumer's dedup primitive (DynamoDB conditional write on a derived hash) silently drops records when the hash collides. No error, no alarm; just a small consistent gap. This is the most common silent-failure pattern at production scale.
- C) Wrong. Clock skew affects ordering and windowing, not record count. If revenue is off, the issue is records-not-counted, not records-counted-in-wrong-bucket.
- D) Wrong. Batching delays records but does not drop them. Eventually they land.

</details>

**Question 2.** You discover a consumer Lambda that calls `dynamodb.put_item` with `ConditionExpression="attribute_not_exists(event_id)"`. The condition keys on a hash you compute from `(event_timestamp, customer_id)`. The producer already emits a unique `event_id` on every record. The smallest fix to eliminate the false-positive duplicate rejections is:

A) Remove the conditional write; trust the producer to never repeat.

B) Use the producer's `event_id` as the conditional-write key instead of the derived hash.

C) Increase the Lambda's reserved concurrency to spread the load and reduce collisions.

D) Switch DynamoDB to provisioned capacity to avoid throttling-related rejections.

<details><summary>Show answer</summary>

**B**.

- A) Wrong. Dropping the conditional write removes the idempotency guarantee. The producer may retry due to network errors and the consumer might process the same record twice; you traded one bug for another.
- B) Right. The bug is that the consumer recomputes the unique key in a lossy way. The producer's `event_id` is already unique. Using it directly removes the collision class entirely. Minimal change, correct outcome.
- C) Wrong. Concurrency does not change the collision probability. The bug is the key derivation, not the throughput.
- D) Wrong. The rejections are `ConditionalCheckFailedException`, not `ProvisionedThroughputExceededException`. Capacity is unrelated.

</details>
